In [1]:
import os
import json
import ast
import pandas as pd

from datasets import load_dataset

from langchain_text_splitters  import RecursiveCharacterTextSplitter

from openai import OpenAI

import chromadb

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
ds = load_dataset("allenai/qasper")

dfs = []

for split_name in ds.keys():

    df = ds[split_name].to_pandas()

    df["split"] = split_name

    dfs.append(df)

merged_df = pd.concat(dfs, ignore_index=True)

print("Total rows:", len(merged_df))

merged_df.head()

# Create dataframe subsets from merged_df

df_50 = merged_df.iloc[:50].copy()
df_100 = merged_df.iloc[:100].copy()
df_250 = merged_df.iloc[:250].copy()
df_500 = merged_df.iloc[:500].copy()
df_750 = merged_df.iloc[:750].copy()
df_1000 = merged_df.iloc[:1000].copy()
df_1500 = merged_df.iloc[:1500].copy()

# Print lengths of all subsets

print("Length of df_50   :", len(df_50))
print("Length of df_100  :", len(df_100))
print("Length of df_250  :", len(df_250))
print("Length of df_500  :", len(df_500))
print("Length of df_750  :", len(df_750))
print("Length of df_1000 :", len(df_1000))
print("Length of df_1500 :", len(df_1500))

Total rows: 1585
Length of df_50   : 50
Length of df_100  : 100
Length of df_250  : 250
Length of df_500  : 500
Length of df_750  : 750
Length of df_1000 : 1000
Length of df_1500 : 1500


In [27]:
def build_eval_dataframe(df):

    rows = []

    for doc_idx, row in df.iterrows():

        qas = row["qas"]

        questions = qas["question"]

        answers = qas["answers"]

        for i in range(len(questions)):

            question = str(questions[i])

            answer_group = answers[i]

            final_answer = ""

            try:

                annotations = answer_group["answer"]

                for ann in annotations:

                    free_form = ann["free_form_answer"]

                    if free_form is not None:

                        free_form = str(free_form).strip()

                        if free_form != "":

                            final_answer = free_form

                            break

                    extractive = ann["extractive_spans"]

                    if len(extractive) > 0:

                        final_answer = " ".join(
                            [str(x) for x in extractive]
                        )

                        break

            except Exception:
                pass

            rows.append({

                "doc_id": doc_idx,

                "question": question,

                "answer": final_answer

            })

    return pd.DataFrame(rows)

In [ ]:
eval_50 = build_eval_dataframe(df_50)
eval_50 = eval_50[eval_50["answer"] != ""]

eval_100 = build_eval_dataframe(df_100)
eval_100 = eval_100[eval_100["answer"] != ""]

eval_250 = build_eval_dataframe(df_250)
eval_250 = eval_250[eval_250["answer"] != ""]

eval_500 = build_eval_dataframe(df_500)
eval_500 = eval_500[eval_500["answer"] != ""]

eval_750 = build_eval_dataframe(df_750)
eval_750 = eval_750[eval_750["answer"] != ""]

eval_1000 = build_eval_dataframe(df_1000)
eval_1000 = eval_1000[eval_1000["answer"] != ""]

eval_1500 = build_eval_dataframe(df_1500)
eval_1500 = eval_1500[eval_1500["answer"] != ""]

In [166]:
# ============================================================
# LLM-BASED QUESTION SELECTION
# Select BEST 25 benchmark questions
# ============================================================

def select_best_questions(eval_df):

    selected_rows = []

    for idx, row in eval_df.iterrows():

        question = row["question"]

        answer = row["answer"]

        prompt = f"""
You are selecting benchmark questions for evaluating:

1. RAG systems
2. Concept-centric WikiLLM systems

QUESTION:
{question}

ANSWER:
{answer}

Evaluate whether this question is GOOD for benchmarking.

Prefer questions that require:
- multi-hop reasoning
- semantic understanding
- concept relationships
- retrieval quality
- synthesis across concepts
- technical detail
- contextual understanding

Avoid questions that are:
- yes/no only
- trivial
- too short
- purely factual lookup
- answerable without retrieval

Return ONLY JSON.

FORMAT:
{{
  "score": 1-10,
  "reason": "short reason"
}}
"""

        try:

            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                response_format={
                    "type": "json_object"
                }
            )

            result = json.loads(
                response.choices[0].message.content
            )

            score = result["score"]

            reason = result["reason"]

        except Exception as e:

            print("ERROR:", e)

            score = 0

            reason = str(e)

        selected_rows.append({

            "question": question,

            "answer": answer,

            "benchmark_score": score,

            "selection_reason": reason

        })

    scored_df = pd.DataFrame(selected_rows)

    scored_df = scored_df.sort_values(
        by="benchmark_score",
        ascending=False
    )

    return scored_df.head(25).reset_index(drop=True)

In [167]:
eval_50_sample = select_best_questions(eval_50)

eval_100_sample = select_best_questions(eval_100)

eval_250_sample = select_best_questions(eval_250)

eval_500_sample = select_best_questions(eval_500)

eval_750_sample = select_best_questions(eval_750)

eval_1000_sample = select_best_questions(eval_1000)

eval_1500_sample = select_best_questions(eval_1500)

In [170]:
import tiktoken

encoding = tiktoken.encoding_for_model(
    "gpt-5-nano"
)

def count_tokens(text):

    return len(
        encoding.encode(text)
    )

In [171]:
def generate_rag_answer(question, collection, top_k=30):

    query_embedding = get_embedding(question)

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    retrieved_chunks = results["documents"][0]

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
Answer the question using ONLY the provided context.

QUESTION:
{question}

CONTEXT:
{context}

ANSWER:
"""

    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    return answer, retrieved_chunks

In [172]:
def add_rag_results(eval_df, collection):

    rag_answers = []

    rag_tokens = []

    for idx, row in eval_df.iterrows():

        question = row["question"]

        try:

            generated_answer, retrieved_chunks = generate_rag_answer(
                question,
                collection,
                top_k=30
            )

            context = "\n\n".join(retrieved_chunks)

            full_prompt = f"""
Answer the question using ONLY the provided context.

QUESTION:
{question}

CONTEXT:
{context}

ANSWER:
"""

            total_tokens = (
                count_tokens(full_prompt)
                + count_tokens(generated_answer)
            )

        except Exception as e:

            print("ERROR:", e)

            generated_answer = ""

            total_tokens = 0

        rag_answers.append(generated_answer)

        rag_tokens.append(total_tokens)

    eval_df = eval_df.copy()

    eval_df["rag_answer"] = rag_answers

    eval_df["rag_tokens"] = rag_tokens

    return eval_df

In [173]:
def load_rag_collection(scale_name):

    chroma_client = chromadb.PersistentClient(
        path=f"benchmark/rag/{scale_name}/chroma_db"
    )

    collection = chroma_client.get_collection(
        name="rag_collection"
    )

    return collection

def get_embedding(text):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    return response.data[0].embedding

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [174]:
collection_50 = load_rag_collection("docs_50")

collection_100 = load_rag_collection("docs_100")

collection_250 = load_rag_collection("docs_250")

collection_500 = load_rag_collection("docs_500")

collection_750 = load_rag_collection("docs_750")

collection_1000 = load_rag_collection("docs_1000")

collection_1500 = load_rag_collection("docs_1500")

In [175]:
eval_50_sample = add_rag_results(eval_50_sample, collection_50)

eval_100_sample = add_rag_results(eval_100_sample, collection_100)

eval_250_sample = add_rag_results(eval_250_sample, collection_250)

eval_500_sample = add_rag_results(eval_500_sample, collection_500)

eval_750_sample = add_rag_results(eval_750_sample, collection_750)

eval_1000_sample = add_rag_results(eval_1000_sample, collection_1000)

eval_1500_sample = add_rag_results(eval_1500_sample, collection_1500)

In [176]:
def load_wikillm_collection(scale_name):

    chroma_client = chromadb.PersistentClient(
        path=f"benchmark/wikillm/{scale_name}/vector_db"
    )

    collection = chroma_client.get_collection(
        name="wikillm_index"
    )

    return collection

wikillm_50 = load_wikillm_collection("docs_50")

wikillm_100 = load_wikillm_collection("docs_100")

wikillm_250 = load_wikillm_collection("docs_250")

wikillm_500 = load_wikillm_collection("docs_500")

wikillm_750 = load_wikillm_collection("docs_750")

wikillm_1000 = load_wikillm_collection("docs_1000")

wikillm_1500 = load_wikillm_collection("docs_1500")

In [177]:
# ============================================================
# EXTRACT SLUG
# ============================================================

def extract_slug(text):

    match = re.search(r"\[\[(.*?)\]\]", text)

    if match:
        return match.group(1)

    return None


# ============================================================
# LOAD PAGE
# ============================================================

def load_page(scale_name, slug):

    page_path = (
        f"benchmark/wikillm/{scale_name}/pages/{slug}.md"
    )

    if not os.path.exists(page_path):

        return None

    with open(page_path, "r", encoding="utf-8") as f:

        return f.read()


# ============================================================
# EXTRACT RELATED CONCEPTS
# ============================================================

def extract_related_concepts(content):

    related = re.findall(
        r"\[\[(.*?)\]\]",
        content
    )

    return list(set(related))


# ============================================================
# BUILD MULTI-HOP CONTEXT
# ============================================================

def build_wikillm_context(
    scale_name,
    retrieved_slugs,
    max_related=10
):

    visited = set()

    context_parts = []

    def add_page(slug, hop=0):

        if slug in visited:
            return

        visited.add(slug)

        content = load_page(scale_name, slug)

        if content is None:
            return

        context_parts.append(
            f"\n\n===== CONCEPT: {slug} =====\n\n"
        )

        context_parts.append(content)

        # ONLY 1-HOP EXPANSION
        if hop >= 1:
            return

        related = extract_related_concepts(content)

        for rel_slug in related[:max_related]:

            add_page(rel_slug, hop + 1)

    for slug in retrieved_slugs:

        add_page(slug)

    return "\n".join(context_parts)


# ============================================================
# RERANK RETRIEVED CONCEPTS
# ============================================================

def rerank_concepts(
    question,
    retrieved_indexes
):

    prompt = f"""
Question:
{question}

Retrieved Concepts:
{retrieved_indexes}

Select the TOP 5 most relevant concepts.

Return ONLY a Python list of concept slugs.

Example:
["transformer", "self-attention"]
"""

    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    text = response.choices[0].message.content

    slugs = re.findall(
        r'"(.*?)"',
        text
    )

    return slugs[:5]


# ============================================================
# GENERATE WIKILLM ANSWER
# ============================================================

def generate_wikillm_answer(
    question,
    collection,
    scale_name,
    top_k=30
):

    # --------------------------------------------------------
    # VECTOR SEARCH
    # --------------------------------------------------------

    query_embedding = get_embedding(question)

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=30
    )

    retrieved_indexes = results["documents"][0]

    # --------------------------------------------------------
    # EXTRACT INITIAL SLUGS
    # --------------------------------------------------------

    retrieved_slugs = []

    for idx_text in retrieved_indexes:

        slug = extract_slug(idx_text)

        if slug is not None:

            retrieved_slugs.append(slug)

    # --------------------------------------------------------
    # LLM RERANKING
    # --------------------------------------------------------

    try:

        retrieved_slugs = rerank_concepts(
            question,
            retrieved_slugs
        )

    except Exception as e:

        print("RERANK ERROR:", e)

    # --------------------------------------------------------
    # BUILD CONTEXT
    # --------------------------------------------------------

    full_context = build_wikillm_context(
        scale_name,
        retrieved_slugs,
        max_related=10
    )

    prompt = f"""
You are answering questions using a concept-centric
knowledge wiki.

Use ONLY the provided context.

If the answer is not present, say:
"Insufficient information in retrieved concepts."

QUESTION:
{question}

CONTEXT:
{full_context}

INSTRUCTIONS:
- combine information across concepts
- use relationships between concepts
- avoid hallucinations
- answer concisely but completely

ANSWER:
"""

    # --------------------------------------------------------
    # GENERATE ANSWER
    # --------------------------------------------------------

    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    # --------------------------------------------------------
    # TOKEN COUNT
    # --------------------------------------------------------

    total_tokens = (
        count_tokens(prompt)
        + count_tokens(answer)
    )

    return answer, total_tokens

In [178]:
import re
def add_wikillm_results(
    eval_df,
    collection,
    scale_name
):

    wikillm_answers = []

    wikillm_tokens = []

    for idx, row in eval_df.iterrows():

        question = row["question"]

        try:

            answer, total_tokens = generate_wikillm_answer(
                question,
                collection,
                scale_name,
                top_k=30
            )

        except Exception as e:

            print("ERROR:", e)

            answer = ""

            total_tokens = 0

        wikillm_answers.append(answer)

        wikillm_tokens.append(total_tokens)

    eval_df = eval_df.copy()

    eval_df["wikillm_answer"] = wikillm_answers

    eval_df["wikillm_tokens"] = wikillm_tokens

    return eval_df

In [181]:
eval_50_sample = add_wikillm_results(eval_50_sample, wikillm_50, "docs_50")

eval_100_sample = add_wikillm_results(eval_100_sample, wikillm_100, "docs_100")

eval_250_sample = add_wikillm_results(eval_250_sample, wikillm_250, "docs_250")

eval_500_sample = add_wikillm_results(eval_500_sample, wikillm_500, "docs_500")

eval_750_sample = add_wikillm_results(eval_750_sample, wikillm_750, "docs_750")

eval_1000_sample = add_wikillm_results(eval_1000_sample, wikillm_1000, "docs_1000")

eval_1500_sample = add_wikillm_results(eval_1500_sample, wikillm_1500, "docs_1500")

In [193]:
# ============================================================
# DETAILED GPT-4o-mini EVALUATION
# ============================================================

def evaluate_answers(eval_df):

    # --------------------------------------------------------
    # RAG
    # --------------------------------------------------------

    rag_factual = []
    rag_semantic = []
    rag_completeness = []
    rag_relevance = []
    rag_hallucination = []
    rag_technical = []

    # --------------------------------------------------------
    # WIKILLM
    # --------------------------------------------------------

    wikillm_factual = []
    wikillm_semantic = []
    wikillm_completeness = []
    wikillm_relevance = []
    wikillm_hallucination = []
    wikillm_technical = []

    # --------------------------------------------------------
    # REASONS
    # --------------------------------------------------------

    rag_reason = []

    wikillm_reason = []

    # --------------------------------------------------------
    # LOOP
    # --------------------------------------------------------

    for idx, row in eval_df.iterrows():

        question = row["question"]

        ground_truth = row["answer"]

        rag_answer = row["rag_answer"]

        wikillm_answer = row["wikillm_answer"]

        prompt = f"""
You are evaluating answers generated by two systems.

QUESTION:
{question}

GROUND TRUTH:
{ground_truth}

RAG ANSWER:
{rag_answer}

WIKILLM ANSWER:
{wikillm_answer}

Evaluate BOTH answers independently.

Metrics (0-10):

1. factual_correctness
2. semantic_similarity
3. completeness
4. relevance
5. hallucination_avoidance
6. technical_accuracy

SCORING GUIDE:
0-2   = very poor
3-4   = poor
5-6   = moderate
7-8   = good
9-10  = excellent

Return ONLY JSON.

FORMAT:
{{
  "rag": {{
    "factual_correctness": 0,
    "semantic_similarity": 0,
    "completeness": 0,
    "relevance": 0,
    "hallucination_avoidance": 0,
    "technical_accuracy": 0,
    "reason": "short explanation"
  }},

  "wikillm": {{
    "factual_correctness": 0,
    "semantic_similarity": 0,
    "completeness": 0,
    "relevance": 0,
    "hallucination_avoidance": 0,
    "technical_accuracy": 0,
    "reason": "short explanation"
  }}
}}
"""

        try:

            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                response_format={
                    "type": "json_object"
                }
            )

            result = json.loads(
                response.choices[0].message.content
            )

            # ------------------------------------------------
            # RAG
            # ------------------------------------------------

            rag = result["rag"]

            rag_factual.append(
                rag["factual_correctness"]
            )

            rag_semantic.append(
                rag["semantic_similarity"]
            )

            rag_completeness.append(
                rag["completeness"]
            )

            rag_relevance.append(
                rag["relevance"]
            )

            rag_hallucination.append(
                rag["hallucination_avoidance"]
            )

            rag_technical.append(
                rag["technical_accuracy"]
            )

            rag_reason.append(
                rag["reason"]
            )

            # ------------------------------------------------
            # WIKILLM
            # ------------------------------------------------

            wiki = result["wikillm"]

            wikillm_factual.append(
                wiki["factual_correctness"]
            )

            wikillm_semantic.append(
                wiki["semantic_similarity"]
            )

            wikillm_completeness.append(
                wiki["completeness"]
            )

            wikillm_relevance.append(
                wiki["relevance"]
            )

            wikillm_hallucination.append(
                wiki["hallucination_avoidance"]
            )

            wikillm_technical.append(
                wiki["technical_accuracy"]
            )

            wikillm_reason.append(
                wiki["reason"]
            )

        except Exception as e:

            print("ERROR:", e)

            rag_factual.append(0)
            rag_semantic.append(0)
            rag_completeness.append(0)
            rag_relevance.append(0)
            rag_hallucination.append(0)
            rag_technical.append(0)
            rag_reason.append(str(e))

            wikillm_factual.append(0)
            wikillm_semantic.append(0)
            wikillm_completeness.append(0)
            wikillm_relevance.append(0)
            wikillm_hallucination.append(0)
            wikillm_technical.append(0)
            wikillm_reason.append(str(e))

    # --------------------------------------------------------
    # ADD COLUMNS
    # --------------------------------------------------------

    eval_df = eval_df.copy()

    # RAG
    eval_df["rag_factual"] = rag_factual
    eval_df["rag_semantic"] = rag_semantic
    eval_df["rag_completeness"] = rag_completeness
    eval_df["rag_relevance"] = rag_relevance
    eval_df["rag_hallucination"] = rag_hallucination
    eval_df["rag_technical"] = rag_technical
    eval_df["rag_reason"] = rag_reason

    # WIKILLM
    eval_df["wikillm_factual"] = wikillm_factual
    eval_df["wikillm_semantic"] = wikillm_semantic
    eval_df["wikillm_completeness"] = wikillm_completeness
    eval_df["wikillm_relevance"] = wikillm_relevance
    eval_df["wikillm_hallucination"] = wikillm_hallucination
    eval_df["wikillm_technical"] = wikillm_technical
    eval_df["wikillm_reason"] = wikillm_reason

    return eval_df

In [194]:
eval_50_sample = evaluate_answers(eval_50_sample)

eval_100_sample = evaluate_answers(eval_100_sample)

eval_250_sample = evaluate_answers(eval_250_sample)

eval_500_sample = evaluate_answers(eval_500_sample)

eval_750_sample = evaluate_answers(eval_750_sample)

eval_1000_sample = evaluate_answers(eval_1000_sample)

eval_1500_sample = evaluate_answers(eval_1500_sample)

In [200]:
summary_table = pd.DataFrame([
    {
        "scale": "eval_50_sample",
        **eval_50_sample.mean(numeric_only=True).to_dict()
    },
    {
        "scale": "eval_100_sample",
        **eval_100_sample.mean(numeric_only=True).to_dict()
    },
    {
        "scale": "eval_250_sample",
        **eval_250_sample.mean(numeric_only=True).to_dict()
    },
    {
        "scale": "eval_500_sample",
        **eval_500_sample.mean(numeric_only=True).to_dict()
    },
    {
        "scale": "eval_750_sample",
        **eval_750_sample.mean(numeric_only=True).to_dict()
    },
    {
        "scale": "eval_1000_sample",
        **eval_1000_sample.mean(numeric_only=True).to_dict()
    },
    {
        "scale": "eval_1500_sample",
        **eval_1500_sample.mean(numeric_only=True).to_dict()
    }
])

summary_table = summary_table.round(2)

In [202]:
summary_table[['scale', 'rag_tokens', 'wikillm_tokens', 'rag_factual', 'rag_semantic', 'rag_completeness',
       'rag_relevance', 'rag_hallucination', 'rag_technical',
       'wikillm_factual', 'wikillm_semantic', 'wikillm_completeness',
       'wikillm_relevance', 'wikillm_hallucination', 'wikillm_technical']]

,scale,rag_tokens,wikillm_tokens,rag_factual,rag_semantic,rag_completeness,rag_relevance,rag_hallucination,rag_technical,wikillm_factual,wikillm_semantic,wikillm_completeness,wikillm_relevance,wikillm_hallucination,wikillm_technical
0,eval_50_sample,4547.56,2863.88,7.56,6.81,6.38,7.88,8.19,7.62,7.56,6.81,7.25,8.00,8.31,7.44
1,eval_100_sample,4383.77,4189.31,7.85,7.00,6.69,8.31,8.31,7.69,6.77,6.15,6.69,7.23,7.38,6.77
2,eval_250_sample,4496.46,6527.92,8.00,7.23,7.00,8.23,8.54,7.77,7.85,6.92,7.31,8.15,8.38,7.69
3,eval_500_sample,4722.35,11112.85,7.75,7.00,6.85,8.20,8.35,7.80,8.05,7.40,7.60,8.55,8.45,8.15
4,eval_750_sample,4634.80,16030.35,7.90,7.10,6.70,7.90,8.45,7.65,8.10,7.50,8.10,8.45,8.55,8.15
5,eval_1000_sample,4572.18,22629.88,8.00,7.29,6.71,8.24,8.59,7.76,8.29,7.71,8.06,8.53,8.71,8.24
6,eval_1500_sample,4601.74,29066.11,7.53,6.79,6.42,7.79,8.11,7.21,7.84,7.26,7.53,8.16,8.21,7.79
